# DS14 · SVD, PCA, compression, and train-only representations

<!-- paper-first -->
### Research question

**Reading:** [PM02](../../curriculum/papers/modeling.md#pm02), [PM03](../../curriculum/papers/modeling.md#pm03). Review the assigned figure or result before starting the lesson.

**Question:** Does preserving variation or representational structure guarantee preserving information for the research target?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Format:** 75–100 minutes of guided work, plus 30–60 minutes in the assigned existing course material. **Prerequisite:** the foundations notebooks; follow this strand in order. The core exercise is synthetic, offline, and independently runnable. It demonstrates mechanics, not a validated participant-data analysis.

## Existing course material

Read [Neuromatch W1D4: geometric representation and dimensionality reduction](https://github.com/NeuromatchAcademy/course-content/blob/44634e960df7a14cd0bf7398187f2d209d26b0e8/tutorials/W1D4_DimensionalityReduction/W1D4_Tutorial1.ipynb). Use the indicated topic, then return here to apply it to a neuroimaging question. Berkeley material is linked in its original form, not adapted or redistributed; its CC BY-NC-ND terms remain upstream. Neuromatch material is CC BY 4.0 with separately licensed software; selected unmodified copies live in `third_party/data_science`. The explanation and dataset below are original.

## Understand the transformation

Principal component analysis chooses orthogonal directions capturing variance after a specified centering and scaling convention. Singular value decomposition supplies a stable numerical route. Rows must have a meaning: in one application they are participants with regional features; in another they are time points with voxel features. Changing that orientation changes the question.

Retaining fewer components compresses the data. The explained-variance ratio measures variance retained under the chosen representation, not the fraction of biologically meaningful signal retained. Scanner effects, motion, and other nuisance factors can dominate high-variance directions. Conversely, a predictive effect may live in a low-variance direction. A component's sign is arbitrary, so a sign flip across software runs is not necessarily a changed scientific result.

For prediction, PCA belongs inside the training pipeline and each cross-validation fold. Fitting it on all participants leaks the test feature distribution even without labels. For descriptive analysis of an entire fixed cohort, the aim is different; state that target explicitly rather than claiming out-of-sample evaluation.

The exercise creates three correlated features with essentially one signal direction. We fit PCA only on the training rows, reconstruct all features with either one or three components, and compare reconstruction error. Then introduce a large shift in held-out data to show that a full-data fitted center changes. An AI explanation should connect every matrix dimension to participants and features before naming the components.

## AI-guided prediction

First answer in your own words; then send this to Goose/Ollama or ChatGPT:

> Ask whether rows are participants or time points. Fit PCA on training rows only, compare one-component and full reconstruction, and explain the arbitrary sign and meaning of retained variance. Do not equate variance explained with causal relevance.

Use the model as a tutor and snippet writer. Require it to name the axes, units, fitting population, expected output, and one failure check. A code cell that runs is not proof that it answers the scientific question. Keep raw data unchanged and save your actual settings.

## Experiment

Compare one- and three-component reconstruction error. Deliberate error: fit PCA on a test set with a shifted mean and silently use that updated representation in evaluation.

Run the following cells in order. Before each, predict what should remain unchanged and what should differ. The assertions test specific mathematical or bookkeeping properties, not clinical validity.

In [1]:
import numpy as np
from sklearn.decomposition import PCA
rng=np.random.default_rng(314); latent=rng.normal(size=(120,1))
X=latent@np.array([[1.,2.,-1.]])+rng.normal(0,.1,(120,3))
train=X[:90]; test=X[90:]+10
p1=PCA(n_components=1).fit(train); p3=PCA(n_components=3).fit(train)
r1=p1.inverse_transform(p1.transform(train)); r3=p3.inverse_transform(p3.transform(train))
assert np.mean((train-r3)**2)<1e-20
assert np.mean((train-r1)**2)>np.mean((train-r3)**2)
leaky=PCA(1).fit(np.vstack([train,test]))
assert np.linalg.norm(leaky.mean_-p1.mean_)>1
print('One-component variance ratio:',p1.explained_variance_ratio_)
print('Train-only / leaky centers:',p1.mean_,leaky.mean_)

One-component variance ratio: [0.99718575]
Train-only / leaky centers: [ 0.05051843  0.10376344 -0.03590799] [2.45918307 2.42404833 2.54542851]


## Explain, break, transfer

1. Save an input → operation → output diagram and state what information was lost.
2. Make the specified wrong choice above. Compare its result with the reference checks; explain why the misleading result is possible.
3. Work through the assigned upstream chapter's example using its own environment or hosted reader. Record one difference between its data and a participant/voxel/time-series dataset.
4. Ask the AI for a short application to a real imaging table, but do not run it until participant identifiers, units, missingness, and any training/test boundary are explicit. Never infer those properties from the column names alone.

**Evidence to submit:** one labeled result, the changed parameter, a failure diagnosis, and a five-sentence interpretation that separates a computational check from the research claim. Explain the result without looking at the model's wording.

<details><summary>Instructor check / answer guide</summary>

Full reconstruction is numerically exact for this complete three-dimensional data. One component loses variation. A full-cohort center is influenced by the shifted test data, demonstrating a fit-boundary violation.

</details>

**Scope:** This local notebook and its numerical checks are part of the executable core. Completion of the external chapter is a learner assignment; its execution is not implied by the local result. No endorsement by the source authors or USC is implied.

### Return to the research question

Revisit [PM02](../../curriculum/papers/modeling.md#pm02), [PM03](../../curriculum/papers/modeling.md#pm03) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
